Imports and paths

In [6]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False
    print("Seaborn not installed. Heatmaps will use matplotlib.")

PROJECT_ROOT = Path(".")
SAVED_DIR = PROJECT_ROOT / "saved_outputs"
SAVED_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT.resolve())
print("Saved outputs:", SAVED_DIR.resolve())

Project root: C:\Users\hbk-2\OneDrive\Documents\GitHub\MRP-Sem_2-Group_8
Saved outputs: C:\Users\hbk-2\OneDrive\Documents\GitHub\MRP-Sem_2-Group_8\saved_outputs


Check available result files

In [7]:
result_extensions = {".csv", ".json", ".npy", ".npz", ".pkl", ".pt", ".png"}

result_files = [
    f for f in PROJECT_ROOT.rglob("*")
    if f.is_file() and f.suffix.lower() in result_extensions
]

print(f"Found {len(result_files)} result files:\n")

for f in result_files:
    print(f)

Found 2 result files:

saved_outputs\perturbations\cifar10_engstrom_linf_eps8_adv.pt
saved_outputs\perturbations\cifar10_standard_linf_eps8_adv.pt


Set file paths

In [8]:
CKA_PATH = SAVED_DIR / "cka_results.csv"
SAT_PATH = SAVED_DIR / "sat_results.csv"
GAAS_PATH = SAVED_DIR / "gaas_results.csv"
TRANSFER_PATH = SAVED_DIR / "transferability_matrix.csv"
ASR_PATH = SAVED_DIR / "asr_results.csv"
MODEL_INFO_PATH = SAVED_DIR / "model_info.csv"

paths = {
    "CKA": CKA_PATH,
    "SAT": SAT_PATH,
    "GAAS": GAAS_PATH,
    "Transferability": TRANSFER_PATH,
    "ASR": ASR_PATH,
    "Model info": MODEL_INFO_PATH,
}

for name, path in paths.items():
    print(f"{name}: {path} | exists = {path.exists()}")

CKA: saved_outputs\cka_results.csv | exists = False
SAT: saved_outputs\sat_results.csv | exists = False
GAAS: saved_outputs\gaas_results.csv | exists = False
Transferability: saved_outputs\transferability_matrix.csv | exists = False
ASR: saved_outputs\asr_results.csv | exists = False
Model info: saved_outputs\model_info.csv | exists = False


Safe loader

In [9]:
def safe_read_csv(path, name):
    path = Path(path)
    
    if path.exists():
        df = pd.read_csv(path)
        print(f"Loaded {name}: {df.shape}")
        display(df.head())
        return df
    
    print(f"Missing {name}: {path}")
    return None


cka_df = safe_read_csv(CKA_PATH, "CKA")
sat_df = safe_read_csv(SAT_PATH, "SAT")
gaas_df = safe_read_csv(GAAS_PATH, "GAAS")
transfer_df = safe_read_csv(TRANSFER_PATH, "Transferability")
asr_df = safe_read_csv(ASR_PATH, "ASR")
model_info_df = safe_read_csv(MODEL_INFO_PATH, "Model info")

Missing CKA: saved_outputs\cka_results.csv
Missing SAT: saved_outputs\sat_results.csv
Missing GAAS: saved_outputs\gaas_results.csv
Missing Transferability: saved_outputs\transferability_matrix.csv
Missing ASR: saved_outputs\asr_results.csv
Missing Model info: saved_outputs\model_info.csv


Convert transferability matrix to long format

In [10]:
def transfer_matrix_to_long(df):
    source_col = df.columns[0]
    
    long_df = df.melt(
        id_vars=source_col,
        var_name="target_model",
        value_name="transfer_asr"
    )
    
    long_df = long_df.rename(columns={source_col: "source_model"})
    return long_df


if transfer_df is not None:
    if "target_model" not in transfer_df.columns:
        transfer_long_df = transfer_matrix_to_long(transfer_df)
    else:
        transfer_long_df = transfer_df.copy()
    
    display(transfer_long_df.head())
else:
    transfer_long_df = None

Layer-wise CKA analysis

In [11]:
def get_layerwise_cka(cka_df):
    required = {
        "source_model",
        "target_model",
        "source_layer",
        "target_layer",
        "cka"
    }
    
    missing = required - set(cka_df.columns)
    
    if missing:
        raise ValueError(f"CKA dataframe is missing columns: {missing}")
    
    layerwise = cka_df[
        cka_df["source_layer"] == cka_df["target_layer"]
    ].copy()
    
    layerwise = layerwise.rename(columns={"source_layer": "layer"})
    
    return layerwise[
        ["source_model", "target_model", "layer", "cka"]
    ]


if cka_df is not None:
    layerwise_cka_df = get_layerwise_cka(cka_df)
    display(layerwise_cka_df.head())
else:
    layerwise_cka_df = None

Plot layer-wise CKA

In [12]:
if layerwise_cka_df is not None:
    for (src, tgt), group in layerwise_cka_df.groupby(["source_model", "target_model"]):
        group = group.sort_values("layer")
        
        plt.figure(figsize=(7, 4))
        plt.plot(group["layer"], group["cka"], marker="o")
        plt.xlabel("Layer depth")
        plt.ylabel("CKA similarity")
        plt.title(f"Layer-wise CKA: {src} vs {tgt}")
        plt.grid(True, alpha=0.3)
        plt.show()

Cross-layer CKA analysis

In [13]:
def plot_cross_layer_cka(cka_df, source_model, target_model):
    pair = cka_df[
        (cka_df["source_model"] == source_model) &
        (cka_df["target_model"] == target_model)
    ].copy()
    
    if pair.empty:
        print(f"No CKA data for {source_model} vs {target_model}")
        return
    
    matrix = pair.pivot_table(
        index="source_layer",
        columns="target_layer",
        values="cka",
        aggfunc="mean"
    )
    
    plt.figure(figsize=(7, 6))
    
    if HAS_SEABORN:
        sns.heatmap(matrix, annot=True, fmt=".2f")
    else:
        plt.imshow(matrix.values, aspect="auto")
        plt.colorbar(label="CKA")
        plt.xticks(range(len(matrix.columns)), matrix.columns)
        plt.yticks(range(len(matrix.index)), matrix.index)
    
    plt.xlabel(f"{target_model} layers")
    plt.ylabel(f"{source_model} layers")
    plt.title(f"Cross-layer CKA: {source_model} vs {target_model}")
    plt.show()

Plot cross-layer CKA for all pairs

In [14]:
if cka_df is not None:
    model_pairs = cka_df[["source_model", "target_model"]].drop_duplicates()
    display(model_pairs)
    
    for _, row in model_pairs.iterrows():
        plot_cross_layer_cka(
            cka_df,
            row["source_model"],
            row["target_model"]
        )

Build pairwise summary

In [15]:
def build_pairwise_summary(
    cka_df=None,
    sat_df=None,
    gaas_df=None,
    transfer_long_df=None,
    model_info_df=None
):
    summary = None
    
    if cka_df is not None:
        same_layer = cka_df[
            cka_df["source_layer"] == cka_df["target_layer"]
        ].copy()
        
        cka_summary = same_layer.groupby(
            ["source_model", "target_model"]
        ).agg(
            mean_cka=("cka", "mean"),
            max_cka=("cka", "max")
        ).reset_index()
        
        best_layers = (
            same_layer.sort_values("cka", ascending=False)
            .groupby(["source_model", "target_model"])
            .first()
            .reset_index()
            [["source_model", "target_model", "source_layer"]]
            .rename(columns={"source_layer": "best_layer"})
        )
        
        cka_summary = cka_summary.merge(
            best_layers,
            on=["source_model", "target_model"],
            how="left"
        )
        
        summary = cka_summary
    
    if sat_df is not None:
        sat_temp = sat_df[["source_model", "target_model", "sat"]].copy()
        summary = sat_temp if summary is None else summary.merge(
            sat_temp,
            on=["source_model", "target_model"],
            how="outer"
        )
    
    if gaas_df is not None:
        gaas_temp = gaas_df[["source_model", "target_model", "gaas"]].copy()
        summary = gaas_temp if summary is None else summary.merge(
            gaas_temp,
            on=["source_model", "target_model"],
            how="outer"
        )
    
    if transfer_long_df is not None:
        transfer_temp = transfer_long_df[
            ["source_model", "target_model", "transfer_asr"]
        ].copy()
        
        summary = transfer_temp if summary is None else summary.merge(
            transfer_temp,
            on=["source_model", "target_model"],
            how="outer"
        )
    
    if model_info_df is not None and summary is not None:
        src_info = model_info_df.rename(columns={
            "model": "source_model",
            "architecture": "source_architecture",
            "training_type": "source_training_type"
        })
        
        tgt_info = model_info_df.rename(columns={
            "model": "target_model",
            "architecture": "target_architecture",
            "training_type": "target_training_type"
        })
        
        summary = summary.merge(src_info, on="source_model", how="left")
        summary = summary.merge(tgt_info, on="target_model", how="left")
    
    return summary

Create and save pairwise summary

In [16]:
pairwise_df = build_pairwise_summary(
    cka_df=cka_df,
    sat_df=sat_df,
    gaas_df=gaas_df,
    transfer_long_df=transfer_long_df,
    model_info_df=model_info_df
)

if pairwise_df is not None:
    display(pairwise_df.head())
    
    output_path = SAVED_DIR / "pairwise_summary.csv"
    pairwise_df.to_csv(output_path, index=False)
    
    print("Saved:", output_path)
else:
    print("No pairwise summary created yet.")

No pairwise summary created yet.


Correlation report

In [17]:
def correlation_report(df, x_cols, y_col):
    rows = []
    
    for x_col in x_cols:
        if x_col in df.columns and y_col in df.columns:
            temp = df[[x_col, y_col]].dropna()
            
            if len(temp) >= 2:
                pearson = temp[x_col].corr(temp[y_col], method="pearson")
                spearman = temp[x_col].corr(temp[y_col], method="spearman")
                
                rows.append({
                    "x_metric": x_col,
                    "y_metric": y_col,
                    "n_pairs": len(temp),
                    "pearson": pearson,
                    "spearman": spearman
                })
    
    return pd.DataFrame(rows)


if pairwise_df is not None:
    corr_df = correlation_report(
        pairwise_df,
        x_cols=["mean_cka", "max_cka", "sat", "gaas"],
        y_col="transfer_asr"
    )
    
    display(corr_df)
    
    corr_output_path = SAVED_DIR / "correlation_summary.csv"
    corr_df.to_csv(corr_output_path, index=False)
    
    print("Saved:", corr_output_path)

Scatter plots

In [18]:
def scatter_metric(df, x, y):
    temp = df[[x, y]].dropna()
    
    if temp.empty:
        print(f"No data for {x} vs {y}")
        return
    
    plt.figure(figsize=(6, 4))
    plt.scatter(temp[x], temp[y])
    plt.xlabel(x)
    plt.ylabel(y)
    plt.title(f"{x} vs {y}")
    plt.grid(True, alpha=0.3)
    plt.show()


if pairwise_df is not None:
    for metric in ["mean_cka", "max_cka", "sat", "gaas"]:
        if metric in pairwise_df.columns and "transfer_asr" in pairwise_df.columns:
            scatter_metric(pairwise_df, metric, "transfer_asr")

Group by training type

In [19]:
if pairwise_df is not None:
    if {"source_training_type", "target_training_type"}.issubset(pairwise_df.columns):
        
        pairwise_df["pair_training_type"] = (
            pairwise_df["source_training_type"].astype(str) +
            " → " +
            pairwise_df["target_training_type"].astype(str)
        )
        
        metric_cols = [
            c for c in ["mean_cka", "max_cka", "sat", "gaas", "transfer_asr"]
            if c in pairwise_df.columns
        ]
        
        training_summary = (
            pairwise_df
            .groupby("pair_training_type")[metric_cols]
            .mean()
            .reset_index()
        )
        
        display(training_summary)
        
        training_output_path = SAVED_DIR / "training_choice_summary.csv"
        training_summary.to_csv(training_output_path, index=False)
        
        print("Saved:", training_output_path)
    
    else:
        print("Missing training type columns.")
        print("Create model_info.csv with: model, architecture, training_type")

Training-type plots

In [20]:
if pairwise_df is not None and "pair_training_type" in pairwise_df.columns:
    
    metric_cols = [
        c for c in ["mean_cka", "sat", "gaas", "transfer_asr"]
        if c in pairwise_df.columns
    ]
    
    for metric in metric_cols:
        plot_df = (
            pairwise_df
            .groupby("pair_training_type")[metric]
            .mean()
            .sort_values()
        )
        
        plt.figure(figsize=(8, 4))
        plot_df.plot(kind="bar")
        plt.ylabel(metric)
        plt.title(f"Average {metric} by training pair type")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()

Load transferable example CKA

In [21]:
TRANSFER_EXAMPLE_CKA_PATH = SAVED_DIR / "transferable_example_cka.csv"

example_cka_df = safe_read_csv(
    TRANSFER_EXAMPLE_CKA_PATH,
    "Transferable example CKA"
)

if example_cka_df is not None:
    same_layer_examples = example_cka_df[
        example_cka_df["source_layer"] == example_cka_df["target_layer"]
    ].copy()
    
    example_summary = (
        same_layer_examples
        .groupby(["source_model", "target_model", "source_layer", "transferable"])["cka"]
        .mean()
        .reset_index()
    )
    
    display(example_summary.head())

Missing Transferable example CKA: saved_outputs\transferable_example_cka.csv


Plot transferable vs non-transferable divergence

In [22]:
if example_cka_df is not None:
    
    for (src, tgt), group in example_summary.groupby(["source_model", "target_model"]):
        plt.figure(figsize=(7, 4))
        
        for transferable_value, g in group.groupby("transferable"):
            label = "Transferable" if transferable_value == 1 else "Non-transferable"
            g = g.sort_values("source_layer")
            
            plt.plot(
                g["source_layer"],
                g["cka"],
                marker="o",
                label=label
            )
        
        plt.xlabel("Layer depth")
        plt.ylabel("Average CKA")
        plt.title(f"Transferable vs non-transferable CKA: {src} vs {tgt}")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()

Generate simple written interpretation

In [23]:
def interpret_correlation(corr_df):
    if corr_df is None or corr_df.empty:
        return "Correlation analysis could not be completed because required result files are missing."
    
    lines = []
    
    for _, row in corr_df.iterrows():
        metric = row["x_metric"]
        pearson = row["pearson"]
        spearman = row["spearman"]
        
        if abs(pearson) >= 0.7:
            strength = "strong"
        elif abs(pearson) >= 0.4:
            strength = "moderate"
        elif abs(pearson) >= 0.2:
            strength = "weak"
        else:
            strength = "very weak"
        
        direction = "positive" if pearson > 0 else "negative"
        
        lines.append(
            f"The relationship between {metric} and transfer ASR shows a "
            f"{strength} {direction} correlation "
            f"(Pearson = {pearson:.3f}, Spearman = {spearman:.3f})."
        )
    
    return "\n".join(lines)


if "corr_df" in globals():
    print(interpret_correlation(corr_df))